# 🗂️ 개인실습 — 로그 파이프라인 적재 (7~8교시) · 정답 모음

> 이 노트북은 `PostgreSQL_3_로그파이프라인_개인실습.ipynb`(문제 노트북)의 **정답**입니다.


---
## ⚙️ 준비 — course_db 만들기 (실행만 하세요)

`db-pg` 컨테이너 안에 `course_db`가 없으면 만듭니다(이미 2~6교시 데모 노트북을 실행했다면 이미 있습니다 — 재실행 안전). 조회 결과를 한 줄씩 출력하는 `run()`도 준비합니다.


In [ ]:
# db-pg 컨테이너 안 PostgreSQL에 'course_db'를 만듭니다 (없을 때만 생성 → 재실행 안전)
# 아래는 Windows(cmd.exe) 기준입니다 — Linux/macOS는 바로 아래 주석 처리된 줄을 대신 쓰세요.
# 📖 문법 참고: psql -tc "SQL"의 -t는 헤더·구분선 없이 결과값만, -c는 그 SQL 하나만 실행하고
#    끝내라는 옵션입니다. findstr /C:"1" >NUL은 출력에서 문자 "1"을 찾되(>NUL로 화면 출력은 숨김)
#    "찾았는지 여부"만 성공/실패로 남깁니다. A || B는 "A가 실패해야 B를 실행"이라는 뜻이므로, 이 줄
#    전체는 "course_db가 이미 있으면 그냥 넘어가고, 없으면 CREATE DATABASE로 새로 만든다"는 뜻입니다.
!docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_database WHERE datname='course_db'" | findstr /C:"1" >NUL || docker exec db-pg psql -U postgres -c "CREATE DATABASE course_db"

# ---- Linux/macOS ----
# !docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_database WHERE datname='course_db'" | grep -q 1 || docker exec db-pg psql -U postgres -c "CREATE DATABASE course_db"


In [ ]:
# 조회 결과를 한 줄씩 출력하는 도우미 (원본 스크립트에는 없지만, 노트북에서 결과를 바로 보기 위해 추가했습니다)
# 📖 문법 참고: def run(sql, params=None)에서 params=None은 "params를 안 주면 기본값 None을
#    쓴다"는 뜻입니다(기본 인자). cur.execute(sql, params)는 params가 None이면 sql을 그대로
#    실행하고, (값1, 값2, ...) 같은 튜플이면 sql 안의 %s 자리에 순서대로 채워 넣고 실행합니다 —
#    뒤에서 run("... WHERE log_date = %s", (LOG_DATE,))처럼 씁니다.
def run(sql, params=None):
    cur.execute(sql, params)
    rows = cur.fetchall()
    for row in rows:
        print(row)
    return rows

print("준비 완료 — course_db 확인/생성됨.")


---
## 🧪 실습 1 (CP1) 정답 — 스키마 설계

`id`는 `SERIAL`(자동 증가), `hour`는 `SMALLINT` + `CHECK (hour BETWEEN 0 AND 23)`, 세 번째 표 이름은 `spike_windows`입니다.


In [ ]:
# 🧪 실습 1 (CP1) — 빈칸(_____)을 채우세요. 원본: lab/load/skeleton/schema.sql
# 📖 문법 참고: os.getenv("KEY", 기본값)는 환경변수 KEY가 있으면 그 값을, 없으면 기본값을
#    돌려줍니다(앞서 본 os.environ["KEY"]와 달리 KeyError 없이 안전하게 기본값으로 넘어감).
#    port는 환경변수에서 문자열로 읽히므로 int(...)로 정수로 바꿔줍니다(psycopg는 포트 번호로
#    정수를 기대합니다).
import os
import psycopg
from dotenv import load_dotenv

load_dotenv()
conn = psycopg.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    dbname=os.getenv("POSTGRES_DB", "course_db"),
    user=os.getenv("POSTGRES_USER", "postgres"),
    password=os.getenv("POSTGRES_PASSWORD", "postgres"),
)
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS spike_windows")
cur.execute("DROP TABLE IF EXISTS latency_stats")
cur.execute("DROP TABLE IF EXISTS hourly_error_stats")

# ① hourly_error_stats : 시간대별 에러 집계
# 📖 문법 참고: SERIAL·SMALLINT·CHECK·UNIQUE의 의미는 2교시(핵심개념 데모)에서 설명한 것과 같습니다
#    (자동증가 PK / 작은 정수 / 값 범위 제한 / 조합 중복 금지). NUMERIC(10,2)는 뒤 ②에서 처음
#    나오는데, "전체 최대 10자리, 그중 소수점 아래 2자리"까지 담는 고정 소수 타입입니다 — 금액·비율처럼
#    float의 오차 없이 정확히 저장하고 싶을 때 씁니다.
cur.execute("""
    CREATE TABLE hourly_error_stats (
        id          SERIAL      PRIMARY KEY,
        log_date    DATE        NOT NULL,
        hour        SMALLINT    NOT NULL
                        CHECK (hour BETWEEN 0 AND 23),
        error_count INTEGER     NOT NULL DEFAULT 0,
        total_count INTEGER     NOT NULL DEFAULT 0,
        UNIQUE (log_date, hour)
    )
""")

# ② latency_stats : 시간대별 응답시간 백분위
cur.execute("""
    CREATE TABLE latency_stats (
        id       SERIAL        PRIMARY KEY,
        log_date DATE          NOT NULL,
        hour     SMALLINT      NOT NULL
                     CHECK (hour BETWEEN 0 AND 23),
        p50      NUMERIC(10,2),
        p95      NUMERIC(10,2),
        p99      NUMERIC(10,2),
        UNIQUE (log_date, hour)
    )
""")

# ③ spike_windows : 에러 급증 구간
cur.execute("""
    CREATE TABLE spike_windows (
        id          SERIAL       PRIMARY KEY,
        log_date    DATE         NOT NULL,
        start_hour  SMALLINT     NOT NULL,
        end_hour    SMALLINT     NOT NULL,
        peak_count  INTEGER      NOT NULL,
        spike_ratio NUMERIC(5,4)
    )
""")
conn.commit()

# 📖 문법 참고: information_schema.tables는 pg_tables와 달리 PostgreSQL 전용이 아니라 여러
#    데이터베이스 제품이 공통으로 지원하는 표준 시스템 뷰입니다(MySQL·SQL Server 등에도 있음).
#    table_name IN (...)은 "이 목록에 있는 이름 중 하나와 같으면"이라는 뜻으로, OR로 여러 번
#    비교하는 것을 줄여 쓴 표현입니다.
run("""
    SELECT table_name
    FROM   information_schema.tables
    WHERE  table_schema = 'public'
      AND  table_name IN ('hourly_error_stats', 'latency_stats', 'spike_windows')
    ORDER BY table_name
""")
# → ('hourly_error_stats',) ('latency_stats',) ('spike_windows',) 3행이 나오면 CP1 통과


---
## 🧪 실습 2 (CP2) 정답 — 트랜잭션으로 적재

`result["log_date"]`, `ON CONFLICT ... DO NOTHING`, `lats.get("p50")`, `spike["start_hour"]`, `conn.commit()`.


In [ ]:
# 🧪 실습 2 (CP2). 원본: lab/load/skeleton/load.py
import json

# 노트북은 lab_dayA/notebooks/에 있다는 전제 — 원본 스크립트 기준 상대경로(lab/load/data/...)에 ../를 붙였습니다.
result_file = "../lab/load/data/result_sample.json"
with open(result_file, encoding="utf-8") as f:
    result = json.load(f)
# 📖 문법 참고: json.load(f)는 열린 파일의 JSON 텍스트를 파이썬 딕셔너리(중첩 가능)로 통째로
#    읽어옵니다. 즉 result는 {"log_date": "...", "hourly_errors": {...}, "latency": {...}, ...}
#    형태의 중첩 딕셔너리이고, 아래에서는 이 안의 값들을 하나씩 꺼내 DB에 넣습니다.

log_date = result["log_date"]
print(f"적재 대상 날짜: {log_date}")

# ① hourly_error_stats 적재
# 📖 문법 참고: dict.items()는 딕셔너리를 (키, 값) 쌍으로 하나씩 돌려줍니다 — for hour_str, stats
#    in ... 로 각 쌍을 두 변수에 나눠 받습니다. int(hour_str)는 JSON의 키는 항상 문자열이라("3")
#    정수 hour 컬럼에 넣기 전에 변환이 필요하기 때문입니다.
# 📖 문법 참고: ON CONFLICT (log_date, hour) DO NOTHING은 "이 INSERT가 UNIQUE(log_date, hour)
#    제약을 위반하면(이미 같은 조합이 있으면) 에러 내지 말고 그냥 아무 것도 하지 말고 넘어가라"는
#    뜻입니다(PostgreSQL의 upsert 문법 중 하나). 이 노트북을 여러 번 실행해도 중복 삽입 에러 없이
#    안전한 이유입니다.
for hour_str, stats in result["hourly_errors"].items():
    cur.execute("""
        INSERT INTO hourly_error_stats (log_date, hour, error_count, total_count)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (log_date, hour) DO NOTHING
    """, (log_date, int(hour_str), stats["error_count"], stats["total_count"]))

# ② latency_stats 적재
# 📖 문법 참고: result.get("latency", {})는 result에 "latency" 키가 없어도 에러 없이 빈 딕셔너리
#    {}를 돌려줍니다(안전한 조회). 반면 lats.get("p50")은 기본값 없이 썼으므로 키가 없으면 None을
#    돌려주는데, 이 None은 psycopg를 거쳐 그대로 SQL의 NULL이 됩니다 — "그 시간대는 p50 값이
#    없었다"는 뜻을 자연스럽게 표현합니다.
for hour_str, lats in result.get("latency", {}).items():
    cur.execute("""
        INSERT INTO latency_stats (log_date, hour, p50, p95, p99)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (log_date, hour) DO NOTHING
    """, (log_date, int(hour_str), lats.get("p50"), lats.get("p95"), lats.get("p99")))

# ③ spike_windows 적재
# 📖 문법 참고: spike["start_hour"]처럼 대괄호로 직접 꺼내는 건 "이 키는 반드시 있어야 한다"는
#    뜻이고(없으면 KeyError로 바로 알아챌 수 있음), spike.get("spike_ratio")처럼 .get()을 쓰는 건
#    "있으면 쓰고 없으면 NULL로 넘어가도 된다"는 뜻입니다 — 같은 딕셔너리 조회라도 그 값이
#    필수인지 선택인지에 따라 다른 방식을 쓴 것입니다.
for spike in result.get("spike_windows", []):
    cur.execute("""
        INSERT INTO spike_windows (log_date, start_hour, end_hour, peak_count, spike_ratio)
        VALUES (%s, %s, %s, %s, %s)
    """, (log_date, spike["start_hour"], spike["end_hour"], spike["peak_count"], spike.get("spike_ratio")))

conn.commit()

run("""
    SELECT 'hourly_error_stats' AS 테이블, COUNT(*) AS 행수 FROM hourly_error_stats
    UNION ALL
    SELECT 'latency_stats',  COUNT(*) FROM latency_stats
    UNION ALL
    SELECT 'spike_windows',  COUNT(*) FROM spike_windows
""")
# → hourly_error_stats 24, latency_stats 24, spike_windows 4 가 나오면 CP2 통과
# 참고: 교안 7교시 본문은 latency_stats를 21행으로 설명하지만, 실제 result_sample.json에는
#       24시간 모두 latency 값이 있어 24행이 정상입니다(교안 서술이 실제 데이터와 다른 부분입니다).


---
## 🧪 실습 3 (CP3) 정답 — 에러율 상위 5개

`NULLIF(total_count, 0)`, `ORDER BY error_rate_pct DESC`, `LIMIT 5`.


In [ ]:
# 🧪 실습 3 (CP3) 정답
LOG_DATE = "2016-11-09"

# 📖 문법 참고: error_count::NUMERIC은 정수 컬럼을 NUMERIC(소수 가능한 타입)으로 캐스팅합니다 —
#    안 하면 정수÷정수는 정수 나눗셈이 되어 소수점이 날아갑니다. NULLIF(total_count, 0)는
#    total_count가 0이면 NULL을, 아니면 total_count 값을 그대로 돌려줍니다 — 나눗셈에서 분모로
#    쓰면 "0으로 나누기" 에러 대신 결과가 NULL이 되게 만드는 안전장치입니다. ROUND(값, 2)는
#    소수점 둘째 자리까지 반올림합니다. run(sql, (LOG_DATE,))처럼 튜플에 값이 하나뿐이어도
#    (LOG_DATE,) 처럼 쉼표를 꼭 붙여야 파이썬이 튜플로 인식합니다.
run("""
    SELECT
        hour,
        error_count,
        total_count,
        ROUND(error_count::NUMERIC / NULLIF(total_count, 0) * 100, 2) AS error_rate_pct
    FROM   hourly_error_stats
    WHERE  log_date = %s
    ORDER BY error_rate_pct DESC
    LIMIT 5
""", (LOG_DATE,))
# → 3시(약 12.77%)가 1위로 나오면 CP3 통과. 2~5위는 데이터 그대로 실측한 값이 나옵니다
#   (교안 8교시 본문의 예시 표(10·4·6시)는 실제 result_sample.json과 다른 값입니다 — 이 노트북은 실제 데이터 기준입니다).


---
## 🧪 실습 4 (CP4) 정답 — JOIN

`JOIN`(=`INNER JOIN`), `s.end_hour`.


In [ ]:
# 🧪 실습 4 (CP4) 정답
# 📖 문법 참고: 지금까지의 JOIN은 ON a.id = b.id처럼 "정확히 같은 값" 기준이었는데, 이번엔
#    ON ... AND l.hour BETWEEN s.start_hour AND s.end_hour로 "범위 안에 들어오면 매칭"입니다 —
#    급증 구간(start_hour~end_hour) 안에 속하는 latency_stats 행을 찾아 연결하는 것입니다.
#    BETWEEN a AND b는 "a 이상 b 이하"(양 끝 포함)라는 뜻입니다.
run("""
    SELECT
        s.start_hour,
        s.end_hour,
        s.peak_count,
        ROUND(s.spike_ratio * 100, 2) AS spike_rate_pct,
        l.hour,
        l.p50,
        l.p95,
        l.p99
    FROM   spike_windows s
    JOIN   latency_stats l
        ON s.log_date = l.log_date
       AND l.hour BETWEEN s.start_hour AND s.end_hour
    WHERE  s.log_date = %s
    ORDER BY s.start_hour, l.hour
""", (LOG_DATE,))
# → 3·14·19·22시 4행이 나오면 CP4 통과
#   (교안 8교시 본문의 예시(0·3~4·3~4·14·21시, 5행)는 실제 result_sample.json과 다릅니다 —
#    실제 spike_windows는 4개 모두 start_hour=end_hour인 단일 시간대라 1대1로 4행이 나옵니다.)


---
## 정리

네 체크포인트 모두 원본 `lab/load/solution/`의 코드와 동일하게 완성했습니다. 이 노트북을 실행한 뒤 터미널에서 `python lab/load/verify.py`를 실행하면 같은 `course_db`를 보므로 CP1~CP4가 그대로 통과합니다.
